In [ ]:
import mujoco
import mujoco.viewer
import numpy as np
import time

MODEL_PATH = "g1_with_hands.xml"

# Load model
model = mujoco.MjModel.from_xml_path(MODEL_PATH)
data = mujoco.MjData(model)

# --- Controller parameters ---
KP = 100.0
KD = 5.0

# Identify joints (modify according to your XML)
RIGHT_HIP_ROLL = model.joint("right_hip_roll_link").id
LEFT_HIP_ROLL = model.joint("left_hip_roll_link").id
RIGHT_KNEE = model.joint("right_knee_link").id
LEFT_KNEE = model.joint("left_knee_link").id

def pd_control(joint_id, target):
    qpos_addr = model.jnt_qposadr[joint_id]
    qvel_addr = model.jnt_dofadr[joint_id]

    pos = data.qpos[qpos_addr]
    vel = data.qvel[qvel_addr]

    torque = KP * (target - pos) - KD * vel
    return torque

def step_right_motion(t):
    """
    Generates a simple lateral stepping pattern.
    """
    step_amplitude = 0.3
    knee_bend = 0.4

    right_target = step_amplitude * np.sin(2 * np.pi * 0.5 * t)
    left_target = -step_amplitude * np.sin(2 * np.pi * 0.5 * t)

    right_knee_target = knee_bend * max(0, np.sin(2 * np.pi * 0.5 * t))
    left_knee_target = knee_bend * max(0, -np.sin(2 * np.pi * 0.5 * t))

    return right_target, left_target, right_knee_target, left_knee_target

# Launch viewer
with mujoco.viewer.launch_passive(model, data) as viewer:
    start = time.time()
    while viewer.is_running():
        t = time.time() - start

        # Get motion targets
        r_hip, l_hip, r_knee, l_knee = step_right_motion(t)

        # Apply torques
        data.ctrl[RIGHT_HIP_ROLL] = pd_control(RIGHT_HIP_ROLL, r_hip)
        data.ctrl[LEFT_HIP_ROLL] = pd_control(LEFT_HIP_ROLL, l_hip)
        data.ctrl[RIGHT_KNEE] = pd_control(RIGHT_KNEE, r_knee)
        data.ctrl[LEFT_KNEE] = pd_control(LEFT_KNEE, l_knee)

        mujoco.mj_step(model, data)
        viewer.sync()